# SAM2 可视化工具演示

本 Notebook 演示如何使用 SAM2 可视化工具包，包括：
1. **图像分割可视化** —— 加载图像、运行推理、多种方式展示结果
2. **视频分割可视化** —— 逐帧传播分割并输出视频
3. **中间张量可视化** —— Image Encoder 特征图、注意力权重、Mask Decoder 输出

## 0. 环境准备

In [ ]:
import os
# Apple MPS 不支持部分算子时回退到 CPU
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# 将项目根目录加入 Python 路径
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"项目根目录：{ROOT}")

In [ ]:
# 选择计算设备
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"使用设备：{device}")

if device.type == "cuda":
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

## 1. 图像分割可视化

### 1.1 加载模型

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# 修改为你实际的权重路径
checkpoint = "checkpoints/sam2.1_hiera_large.pt"
model_cfg  = "configs/sam2.1/sam2.1_hiera_l.yaml"

sam2_model = build_sam2(model_cfg, checkpoint, device=device)
predictor  = SAM2ImagePredictor(sam2_model)
print("模型加载完成")

### 1.2 加载示例图像

In [ ]:
image_path = "notebooks/images/truck.jpg"
image = np.array(Image.open(image_path).convert("RGB"))
print(f"图像尺寸：{image.shape}")

plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title("原始图像")
plt.axis("off")
plt.show()

### 1.3 运行推理（点提示）

In [ ]:
# 前景点（卡车中心）
point_coords = np.array([[500, 375]])
point_labels = np.array([1])          # 1=前景

with torch.inference_mode():
    predictor.set_image(image)
    masks, scores, logits = predictor.predict(
        point_coords=point_coords,
        point_labels=point_labels,
        multimask_output=True,
    )

print(f"掩码数量：{len(masks)}")
print(f"置信度：{scores}")
print(f"掩码形状：{masks.shape}")

### 1.4 可视化分割结果

In [ ]:
from visualizations import ImageVisualizer

viz = ImageVisualizer(alpha=0.5, show_borders=True)

# 显示所有候选掩码
fig = viz.show_segmentation(
    image, masks, scores,
    point_coords=point_coords,
    point_labels=point_labels,
    title="SAM2 图像分割 — 所有候选掩码",
)
plt.show()

In [ ]:
# 显示最佳掩码
fig = viz.show_best_mask(
    image, masks, scores,
    point_coords=point_coords,
    point_labels=point_labels,
    title="最佳分割掩码",
)
plt.show()

In [ ]:
# 三视图对比：原始图像 / 掩码叠加 / 分割边界
fig = viz.show_comparison(
    image, masks, scores,
    point_coords=point_coords,
    point_labels=point_labels,
    title="分割结果三视图对比",
)
plt.show()

### 1.5 边界框提示

In [ ]:
# 使用边界框提示
box = np.array([425, 600, 700, 875])  # [x0, y0, x1, y1]

with torch.inference_mode():
    predictor.set_image(image)
    masks_box, scores_box, _ = predictor.predict(
        box=box,
        multimask_output=False,
    )

fig = viz.show_comparison(
    image, masks_box, scores_box,
    box=box,
    title="边界框提示分割结果",
)
plt.show()

### 1.6 使用工具函数

In [ ]:
from visualizations import overlay_mask_on_image, draw_boundaries, apply_colormap

best_mask = masks[scores.argmax()]

# 彩色叠加
overlaid = overlay_mask_on_image(image, best_mask, color=[255, 100, 0], alpha=0.4)

# 边界绘制
boundary = draw_boundaries(image, best_mask, color=[255, 255, 0], thickness=3)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(image);      axes[0].set_title("原始图像");  axes[0].axis("off")
axes[1].imshow(overlaid);   axes[1].set_title("彩色叠加");  axes[1].axis("off")
axes[2].imshow(boundary);   axes[2].set_title("边界绘制");  axes[2].axis("off")
plt.tight_layout()
plt.show()

## 2. 中间张量可视化

### 2.1 Image Encoder 特征图

In [ ]:
from visualizations import TensorVisualizer

tv = TensorVisualizer(figsize=(14, 8))

# 提取图像特征（set_image 后可从 _features 访问）
with torch.inference_mode():
    predictor.set_image(image)

# backbone 特征图（取最后一层）
if predictor._features is not None:
    # image_embed: (1, C, H, W)
    image_embed = predictor._features["image_embed"]
    print(f"image_embed 形状：{image_embed.shape}")

    fig = tv.show_feature_maps(
        image_embed,
        max_channels=16,
        title="Image Encoder 嵌入特征图（前 16 通道）",
    )
    plt.show()

    fig = tv.show_feature_summary(
        image_embed,
        title="Image Encoder 嵌入特征图统计",
    )
    plt.show()
else:
    print("无法获取图像特征，请先调用 predictor.set_image()")

### 2.2 高分辨率特征图对比

In [ ]:
if predictor._features is not None:
    # high_res_feats: 列表，包含多尺度特征
    high_res_feats = predictor._features.get("high_res_feats", [])
    if high_res_feats:
        features_dict = {f"高分辨率特征 {i}": feat for i, feat in enumerate(high_res_feats)}
        fig = tv.show_encoder_features_comparison(
            features_dict,
            title="多尺度高分辨率特征图对比",
        )
        plt.show()
    else:
        print("未找到高分辨率特征图")

### 2.3 Mask Decoder Logit 可视化

In [ ]:
# logits 由 predictor.predict() 第三个返回值获得
with torch.inference_mode():
    predictor.set_image(image)
    masks_vis, scores_vis, logits_vis = predictor.predict(
        point_coords=np.array([[500, 375]]),
        point_labels=np.array([1]),
        multimask_output=True,
        return_logits=True,
    )

print(f"Logit 形状：{logits_vis.shape}")

fig = tv.show_mask_logits(
    logits_vis,
    threshold=0.0,
    title="Mask Decoder 输出 Logit 与二值掩码",
)
plt.show()

## 3. 视频分割可视化

### 3.1 准备视频帧

In [ ]:
import tempfile, shutil
from pathlib import Path

video_path = "notebooks/videos/bedroom.mp4"

# 将视频帧解压到临时目录（SAM2VideoPredictor 需要图像目录）
tmp_frames_dir = tempfile.mkdtemp(prefix="sam2_demo_")

import cv2
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
frames_rgb = []
idx = 0
while True:
    ret, bgr = cap.read()
    if not ret or idx >= 30:   # 演示只取前 30 帧
        break
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    frames_rgb.append(rgb)
    Image.fromarray(rgb).save(f"{tmp_frames_dir}/{idx:05d}.jpg")
    idx += 1
cap.release()

print(f"已提取 {len(frames_rgb)} 帧到 {tmp_frames_dir}")

### 3.2 初始化 VideoPredictor 并传播分割

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

video_predictor = build_sam2_video_predictor(model_cfg, checkpoint, device=device)

# 提示点（在第 0 帧上标注目标）
point_coords = np.array([[250, 200]])
point_labels = np.array([1])

with torch.inference_mode():
    inference_state = video_predictor.init_state(video_path=tmp_frames_dir)
    video_predictor.reset_state(inference_state)

    _, out_obj_ids, out_mask_logits = video_predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=0,
        obj_id=1,
        points=point_coords,
        labels=point_labels,
    )

    masks_per_frame = []
    for frame_idx, obj_ids, mask_logits in video_predictor.propagate_in_video(inference_state):
        frame_masks = {}
        for oid, logit in zip(obj_ids, mask_logits):
            frame_masks[oid] = (logit[0] > 0.0).cpu().numpy()
        masks_per_frame.append(frame_masks)

print(f"完成 {len(masks_per_frame)} 帧的分割传播")

### 3.3 预览关键帧

In [ ]:
from visualizations import VideoVisualizer

vviz = VideoVisualizer(alpha=0.5)

preview_indices = [0, len(frames_rgb) // 2, len(frames_rgb) - 1]
fig, axes = plt.subplots(len(preview_indices), 2, figsize=(16, 6 * len(preview_indices)))

for row, fi in enumerate(preview_indices):
    if fi >= len(masks_per_frame):
        continue
    rendered = vviz.render_frame(frames_rgb[fi], masks_per_frame[fi])
    axes[row][0].imshow(frames_rgb[fi])
    axes[row][0].set_title(f"原始帧（第 {fi} 帧）", fontsize=13)
    axes[row][0].axis("off")
    axes[row][1].imshow(rendered)
    axes[row][1].set_title(f"分割结果（第 {fi} 帧）", fontsize=13)
    axes[row][1].axis("off")

plt.tight_layout()
plt.show()

### 3.4 输出分割视频

In [ ]:
output_path = "output_visualization/bedroom_segmented.mp4"

vviz.create_segmentation_video(frames_rgb, masks_per_frame, output_path, fps=fps)
print(f"视频已保存：{Path(output_path).resolve()}")

# 清理临时文件
shutil.rmtree(tmp_frames_dir, ignore_errors=True)

## 4. 保存可视化结果

In [ ]:
from pathlib import Path

out_dir = Path("output_visualization")
out_dir.mkdir(exist_ok=True)

# 重新生成图像分割结果并保存
with torch.inference_mode():
    predictor.set_image(image)
    masks_save, scores_save, _ = predictor.predict(
        point_coords=np.array([[500, 375]]),
        point_labels=np.array([1]),
        multimask_output=True,
    )

viz_save = ImageVisualizer(alpha=0.5, show_borders=True)

viz_save.show_segmentation(
    image, masks_save, scores_save,
    title="所有候选掩码",
    save_path=out_dir / "all_masks.png",
)

viz_save.show_comparison(
    image, masks_save, scores_save,
    title="三视图对比",
    save_path=out_dir / "comparison.png",
)

viz_save.save_overlay(image, masks_save, out_dir / "overlay.png", scores=scores_save)

print(f"所有结果已保存到：{out_dir.resolve()}")